<!-- cell-01 -->
# Dictionary Learning - Text-Grounded Semantic Anchoring

Implements Section 7 (Text-Grounded Ground Truth and Semantic Anchoring).

Goal: build a semantic representation $u_j$ for each SAE dictionary atom/feature $j$ from curated natural-language ground-truth explanations, using a biomedical text encoder (MedCPT).

## Key objects (shapes)

Let $d$ be the embedding dimension and $m$ be the SAE dictionary size (latent width). For each SL pair $i$ we have a normalized embedding

$$x_i \in \mathbb{R}^d,$$

and the SAE encoder produces a sparse code

$$z_i = f_\theta(x_i) \in \mathbb{R}^m.$$

Stacking all $N$ samples row-wise gives

$$X \in \mathbb{R}^{N\times d}, \qquad Z = f_\theta(X) \in \mathbb{R}^{N\times m}.$$

In [1]:
# cell-02
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

# Make `src/` importable
REPO_ROOT = Path.cwd().resolve()
SRC_DIR = (REPO_ROOT / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from SAE.SAE_training.model import SAEConfig, SparseAutoencoder
from SAE.SAE_training.utils.data import build_artifacts, extract_concat_matrix
from SAE.SAE_training.utils.semantic import load_encoder, encode_texts
from SAE.LLM_pipeline.utils.general import sanitize_groundtruth_text
from SAE.manifold.utils.projection import encode_dataset


/home/guoyu/SLformer_interpretation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# cell-03
CONFIG_PATH = (REPO_ROOT / "src" / "SAE" / "SAE_training" / "config" / "train_config.yaml").resolve()
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

paths_cfg = cfg["paths"]
scope_cfg = cfg["scope"]
norm_cfg = cfg["normalization"]
model_cfg = SAEConfig(**dict(cfg["model"]))

CANCER = str(scope_cfg["cancer"])
SEED = int(scope_cfg["seed"])
MAX_SAMPLES = scope_cfg["max_samples"]
EPS = float(norm_cfg["eps"])

output_base_dir = Path(paths_cfg["output_base_dir"]).expanduser().resolve()
slformer_output_subdir = str(paths_cfg["slformer_output_subdir"])
groundtruth_dir = Path(paths_cfg["groundtruth_dir"]).expanduser().resolve()
medcpt_query_encoder = str(paths_cfg["medcpt_query_encoder"])

run_name = f"hidden{model_cfg.d_hidden}_gate{model_cfg.gate_weight}_orth{model_cfg.orth_weight}_k{model_cfg.topk}"
run_dir = output_base_dir / slformer_output_subdir / CANCER / run_name
dict_out = run_dir / "dictionary"

print("run_dir:", run_dir)
print("dict_out:", dict_out)
print("groundtruth_dir:", groundtruth_dir)
print("medcpt_query_encoder:", medcpt_query_encoder)

run_dir: /home/guoyu/SLformer_interpretation/output/SAE/slformer/mix/hidden4096_gate0.5_orth1_k64
dict_out: /home/guoyu/SLformer_interpretation/output/SAE/slformer/mix/hidden4096_gate0.5_orth1_k64/dictionary
groundtruth_dir: /home/guoyu/SLformer_interpretation/data/explanation_groundtruth
medcpt_query_encoder: /data/guoyu/HF-models/MedCPT-Query-Encoder


In [3]:
# cell-04
mu = np.load(run_dir / "norm" / "mu.npy")
sigma = np.load(run_dir / "norm" / "sigma.npy")

ckpt = torch.load(run_dir / "final.pt", map_location="cpu")
sae_cfg_loaded = SAEConfig(**ckpt["sae_cfg"])
model = SparseAutoencoder(sae_cfg_loaded)
model.load_state_dict(ckpt["state_dict"])
model.eval()

print("Loaded SAE:", sae_cfg_loaded)

Loaded SAE: SAEConfig(d_in=1024, d_hidden=4096, activation='jumprelu', orth_weight=1, decoder_unit_norm=True, topk=64, gate_weight=0.5, jump_threshold=0.0)


In [4]:
# cell-05
embeddings_pkl = Path(paths_cfg["embeddings_pkl"]).expanduser().resolve()
prediction_csvs = [Path(p).expanduser().resolve() for p in paths_cfg["prediction_csvs"]]
score_col = str(scope_cfg["score_col"])

artifacts = build_artifacts(embeddings_pkl=embeddings_pkl, prediction_csvs=prediction_csvs)
X, y, meta = extract_concat_matrix(
    artifacts,
    cancer=CANCER,
    max_samples=MAX_SAMPLES,
    seed=SEED,
    use_score_col=score_col,
)
Xn = ((X - mu) / (sigma + EPS)).astype(np.float32)

print("Xn:", Xn.shape, "y:", y.shape, "meta:", meta.shape)
meta.head(3)

Xn: (58672, 1024) y: (58672,) meta: (58672, 5)


,primary_gene,partner_gene,cancer,score,fold
0,AKT1,BCL2,KIRC,0.000425,0
1,PTEN,RUNX1,KIRC,0.001516,0
2,HDAC6,SMARCA4,KIRC,0.005020,0


## SAE encoding ($X_n \mapsto Z$)

Given normalized inputs $X_n\in\mathbb{R}^{N\times d}$, the SAE encoder $f_\theta$ is applied row-wise to obtain codes
$$Z = f_\theta(X_n) \in \mathbb{R}^{N\times m}.$$

After encoding, we obtain

$$Z = [z_1^\top;\dots;z_N^\top] \in \mathbb{R}^{N\times m}, \qquad z_i = f_\theta(x_{n,i}).$$

For a **TopK-gated** SAE with parameter $k$, each *row* is sparse:

$$\|z_i\|_0 \le k.$$

Across the full dataset, the set of features that ever activate is the union

$$\mathcal{A} = \{ j \in \{1,\dots,m\} : \exists i\ \text{s.t.}\ z_{i,j} \ne 0 \}.$$

In [5]:
# cell-06
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2048
# Batching only affects throughput/memory; SAE encoding is per-sample in eval mode.

Z = encode_dataset(model, Xn, batch_size=BATCH_SIZE, device=DEVICE)
print("Z:", Z.shape)

Z: (58672, 4096)


<!-- cell-07 -->
## Ground-Truth Text Format (what “text evidence” means here)

Define the curated ground-truth set as

$$\mathcal{G} = \{(g_A^p, g_B^p, t^p)\}_{p=1}^{P},$$

where $(g_A^p,g_B^p)$ is a gene pair and $t^p$ is a natural-language explanation (mechanism summary / curated evidence).


In code we materialize a dictionary
$$\texttt{pair2texts}[(g_A,g_B)] = [t^p,\dots]$$
(a list in case multiple texts exist per pair).

In [6]:
# cell-08
# Read ground-truth texts from folder structure: data/explanation_groundtruth/GENE-GENE/explanation.txt
groundtruth_dir = Path(paths_cfg["groundtruth_dir"]).expanduser().resolve()

pair2texts = {}
for pair_folder in sorted(groundtruth_dir.glob("*-*")):
    explanation_file = pair_folder / "explanation.txt"

    geneA, geneB = pair_folder.name.split("-", 1)
    key = (geneA, geneB)
    text = sanitize_groundtruth_text(explanation_file.read_text(encoding="utf-8"))
    if text:
        pair2texts[key] = [text]

print("ground-truth pairs:", len(pair2texts))
print("ground-truth texts:", sum(len(texts) for texts in pair2texts.values()))

ground-truth pairs: 37
ground-truth texts: 37


<!-- cell-08b -->
## Text encoding ($t \mapsto \phi(t)$, and the matrix $T$)

Collect all curated explanation texts $t^p$ from $\mathcal{G}$, deduplicate them, and embed each text with the biomedical encoder $\phi(\cdot)$ (MedCPT):
$$
\phi(t) \in \mathbb{R}^q.
$$

Stacking embeddings of the $n_{\text{text}}$ unique texts yields a matrix
$$
T = [\phi(t_1)^\top;\dots;\phi(t_{n_{\text{text}}})^\top] \in \mathbb{R}^{n_{\text{text}}\times q}.
$$

The lookup `text2row[t]` is the index mapping $t \mapsto \{1,\dots,n_{\text{text}}\}$ so later we can gather the relevant rows of $T$ for each feature $j$ and average them into $u_j$.

In [7]:
# cell-09
tokenizer, text_model, device = load_encoder(
    medcpt_query_encoder,
    device=("cuda" if torch.cuda.is_available() else "cpu"),
)

all_texts = []
for texts in pair2texts.values():
    all_texts.extend(texts)
all_texts = list(dict.fromkeys(all_texts))

T = encode_texts(tokenizer, text_model, device, all_texts, batch_size=64)
text2row = {t: i for i, t in enumerate(all_texts)}
print("Encoded texts:", T.shape)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Encoded texts: (37, 768)


<!-- cell-09b -->
## `TOP_M` exemplars (what this hyperparameter means)

At this point we have SAE codes $Z\in\mathbb{R}^{N\times m}$. The goal of the exemplar step is: for each feature $j$, find the **$M$ samples where that feature is most active**.

Define an activation magnitude (as used in code):

$$a_{i,j} = |Z_{i,j}|.$$

Then the exemplar index set for feature $j$ is

$$\mathcal{E}_j = \operatorname{TopM}_{i\in\{1,\dots,N\}}\big(a_{i,j}\big),$$

In [8]:
# cell-10
TOP_M = 200

feature_ids = np.arange(Z.shape[1])

feature_exemplars = {}
for j in feature_ids:
    col = np.abs(Z[:, j])
    idx = np.argpartition(-col, TOP_M - 1)[:TOP_M]
    idx = idx[np.argsort(-col[idx])]
    feature_exemplars[int(j)] = idx.astype(int).tolist()

print(f"Prepared top {TOP_M} exemplars for {len(feature_exemplars)} SAE features.")

Prepared top 200 exemplars for 4096 SAE features.


## 1) From exemplars to text sets

For each feature $j$, we already formed its top-activating exemplar indices $\mathcal{E}_j\subset\{1,\dots,N\}$ with $|\mathcal{E}_j|=M$.

Each exemplar index $i\in\mathcal{E}_j$ corresponds to a gene pair $(g_{A,i}, g_{B,i})$ from `meta`. We attach curated texts when available via `pair2texts`:
$$
\mathcal{T}_j = \{\, t : \exists i\in\mathcal{E}_j \text{ s.t. } (g_{A,i}, g_{B,i})\in\mathcal{G} \text{ and } t\in \texttt{pair2texts}[(g_{A,i}, g_{B,i})] \,\}.
$$

So $\mathcal{T}_j$ is the collection (multiset) of all ground-truth explanation texts that are *reachable* from feature $j$ through its exemplars.

## 2) Encode texts with MedCPT

Let the text encoder be
$$\phi(\cdot): \text{text} \to \mathbb{R}^q.$$

If we encode all unique texts into a matrix
$$T \in \mathbb{R}^{n_{\text{text}}\times q},$$
then the vector for a specific feature $j$ is an aggregate (here: mean) over the rows corresponding to texts in $\mathcal{T}_j$:
$$
u_j = \operatorname{Mean}\big(\{\phi(t): t\in\mathcal{T}_j\}\big)
= \frac{1}{|\mathcal{T}_j|}\sum_{t\in\mathcal{T}_j}\phi(t)
\qquad \text{(defined only if }|\mathcal{T}_j|>0\text{)}.
$$

In [9]:
# cell-11
geneA_col = "primary_gene"
geneB_col = "partner_gene"

feature_text_counts = np.zeros((Z.shape[1],), dtype=np.int32)
feature_u = {}

for j, idxs in feature_exemplars.items():
    rows = []
    for i in idxs:
        key = (str(meta.loc[i, geneA_col]), str(meta.loc[i, geneB_col]))
        if key in pair2texts:
            for t in pair2texts[key]:
                rows.append(text2row[t])

    if len(rows) == 0:
        continue

    rows = np.asarray(rows, dtype=np.int64)
    Uj = T[rows].mean(axis=0)
    feature_u[int(j)] = Uj.astype(np.float32)
    feature_text_counts[int(j)] = int(rows.shape[0])

print("Features with semantic vectors:", len(feature_u))

Features with semantic vectors: 306


In [10]:
# cell-12
dict_out.mkdir(parents=True, exist_ok=True)

np.savez(
    dict_out / "feature_semantic_embeddings.npz",
    **{str(k): v for k, v in feature_u.items()},
)

df_counts = pd.DataFrame({"feature": np.arange(Z.shape[1]), "n_texts": feature_text_counts})
df_counts.to_csv(dict_out / "feature_text_counts.csv", index=False)

pd.DataFrame({"text_id": np.arange(len(all_texts)), "text": all_texts}).to_csv(
    dict_out / "groundtruth_text_index.csv", index=False,
)

print("Saved to:", dict_out)

Saved to: /home/guoyu/SLformer_interpretation/output/SAE/slformer/mix/hidden4096_gate0.5_orth1_k64/dictionary
